In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class AttentionGate(nn.Module):

    def __init__(self, F_g, F_l, F_int):

        super().__init__()

        self.Wg = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.Wx = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):

        g1 = self.Wg(g)
        x1 = self.Wx(x)

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        return x * psi


class UNet(nn.Module):

    def __init__(self, n_class=1, dropout_p=0.5):

        super().__init__()

        # # ------------- PRETRAINED ENCODER ------------- 
        self.encoder = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            features_only=True,
            in_chans=1,
            out_indices=(0, 1, 2, 3, 4)
        )

        # EfficientNet-B0 feature channels
        # [16, 24, 40, 112, 320]

        self.drop1 = nn.Dropout2d(p=dropout_p)         # deepest block
        self.drop2 = nn.Dropout2d(p=dropout_p)
        self.drop3 = nn.Dropout2d(p=dropout_p * 0.67)  # ~0.2
        self.drop4 = nn.Dropout2d(p=dropout_p * 0.33)  # ~0.1 — lightest, near output

        
        # ------------- DECODER -------------

        # Decoder Block 1
        self.up1 = nn.ConvTranspose2d(320, 112, kernel_size=2, stride=2)

        self.att1 = AttentionGate(F_g=112, F_l=112, F_int=56)

        self.d11 = nn.Conv2d(224, 112, kernel_size=3, padding=1)
        self.bn_d11 = nn.BatchNorm2d(112)

        self.d12 = nn.Conv2d(112, 112, kernel_size=3, padding=1)
        self.bn_d12 = nn.BatchNorm2d(112)

        # Decoder Block 2
        self.up2 = nn.ConvTranspose2d(112, 40, kernel_size=2, stride=2)

        self.att2 = AttentionGate(F_g=40, F_l=40, F_int=20)

        self.d21 = nn.Conv2d(80, 40, kernel_size=3, padding=1)
        self.bn_d21 = nn.BatchNorm2d(40)

        self.d22 = nn.Conv2d(40, 40, kernel_size=3, padding=1)
        self.bn_d22 = nn.BatchNorm2d(40)

        # Decoder Block 3
        self.up3 = nn.ConvTranspose2d(40, 24, kernel_size=2, stride=2)

        self.att3 = AttentionGate(F_g=24, F_l=24, F_int=12)

        self.d31 = nn.Conv2d(48, 24, kernel_size=3, padding=1)
        self.bn_d31 = nn.BatchNorm2d(24)

        self.d32 = nn.Conv2d(24, 24, kernel_size=3, padding=1)
        self.bn_d32 = nn.BatchNorm2d(24)

        # Decoder Block 4
        self.up4 = nn.ConvTranspose2d(24, 16, kernel_size=2, stride=2)

        self.att4 = AttentionGate(F_g=16, F_l=16, F_int=8)

        self.d41 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.bn_d41 = nn.BatchNorm2d(16)

        self.d42 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_d42 = nn.BatchNorm2d(16)

        # FINAL UPSAMPLING
        self.final_up = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)

        self.d_final1 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_final1 = nn.BatchNorm2d(16)
        self.d_final2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn_final2 = nn.BatchNorm2d(16)

        # OUTPUT
        self.outconv = nn.Conv2d(16, n_class, kernel_size=1)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        # =================================================
        # ENCODER
        # =================================================

        features = self.encoder(x)
    
        x1 = features[0]   # 16 channels
        x2 = features[1]   # 24 channels
        x3 = features[2]   # 40 channels
        x4 = features[3]   # 112 channels
        x5 = features[4]   # 320 channels

        # =================================================
        # DECODER 1
        # =================================================

        d1 = self.up1(x5)
        d1 = F.interpolate(d1, size=x4.shape[-2:], mode='bilinear', align_corners=False)
        x4_att = self.att1(x4, d1)
        d1 = torch.cat([d1, x4_att], dim=1)

        d1 = self.relu(self.bn_d11(self.d11(d1)))
        d1 = self.relu(self.bn_d12(self.d12(d1)))
        d1 = self.drop1(d1)

        # =================================================
        # DECODER 2
        # =================================================

        d2 = self.up2(d1)
        d2 = F.interpolate(d2, size=x3.shape[-2:], mode='bilinear', align_corners=False)
        x3_att = self.att2(x3, d2)
        d2 = torch.cat([d2, x3_att], dim=1)

        d2 = self.relu(self.bn_d21(self.d21(d2)))
        d2 = self.relu(self.bn_d22(self.d22(d2)))
        d2 = self.drop2(d2)

        # =================================================
        # DECODER 3
        # =================================================

        d3 = self.up3(d2)
        d3 = F.interpolate(d3, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        x2_att = self.att3(x2, d3)
        d3 = torch.cat([d3, x2_att], dim=1)

        d3 = self.relu(self.bn_d31(self.d31(d3)))
        d3 = self.relu(self.bn_d32(self.d32(d3)))
        d3 = self.drop3(d3)

        # =================================================
        # DECODER 4
        # =================================================

        d4 = self.up4(d3)
        d4 = F.interpolate(d4, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        x1_att = self.att4(x1, d4)
        d4 = torch.cat([d4, x1_att], dim=1)

        d4 = self.relu(self.bn_d41(self.d41(d4)))
        d4 = self.relu(self.bn_d42(self.d42(d4)))
        d4 = self.drop4(d4)

        # =================================================
        # FINAL UPSAMPLING
        # =================================================

        d4 = self.final_up(d4)
        d4 = self.relu(self.bn_final1(self.d_final1(d4)))
        d4 = self.relu(self.bn_final2(self.d_final2(d4)))

        # =================================================
        # OUTPUT
        # =================================================

        out = self.outconv(d4)

        return out

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import kagglehub
import random


import random
import cv2
import numpy as np

def augment(image, mask):

    # Horizontal flip
    if random.random() > 0.3:
        image = cv2.flip(image, 1)
        mask  = cv2.flip(mask, 1)

    # Vertical flip
    if random.random() > 0.3:
        image = cv2.flip(image, 0)
        mask  = cv2.flip(mask, 0)

    # Rotation
    if random.random() > 0.3:
        angle = random.uniform(-30, 30)
        h, w  = image.shape[:2]
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR)
        mask  = cv2.warpAffine(mask,  M, (w, h), flags=cv2.INTER_NEAREST)

    # Brightness / contrast
    if random.random() > 0.3:
        alpha = 0.7 + random.random() * 0.6   # wider range: 0.7–1.3
        beta  = random.randint(-20, 20)         # stronger shift
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    # Gaussian noise
    if random.random() > 0.3:
        noise = np.random.normal(0, random.uniform(2, 25), image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # Gaussian blur
    if random.random() > 0.3:
        ksize = random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ksize, ksize), 0)

    # Scale jitter — single resize via crop/pad instead of double resize
    if random.random() > 0.3:
        scale = random.uniform(0.8, 1.2)
        h, w  = image.shape[:2]
        new_h, new_w = int(h * scale), int(w * scale)
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask,  (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        # Crop or pad back to original size without a second resize pass
        if scale > 1.0:
            start_y = (new_h - h) // 2
            start_x = (new_w - w) // 2
            image = image[start_y:start_y+h, start_x:start_x+w]
            mask  = mask [start_y:start_y+h, start_x:start_x+w]
        else:
            pad_y = (h - new_h) // 2
            pad_x = (w - new_w) // 2
            image = cv2.copyMakeBorder(image, pad_y, h-new_h-pad_y, pad_x, w-new_w-pad_x,
                                       cv2.BORDER_CONSTANT, value=0)
            mask  = cv2.copyMakeBorder(mask,  pad_y, h-new_h-pad_y, pad_x, w-new_w-pad_x,
                                       cv2.BORDER_CONSTANT, value=0)

    # Elastic deformation — critical for small medical datasets
    if random.random() > 0.5:
        h, w   = image.shape[:2]
        sigma  = random.uniform(6, 10)
        alpha  = random.uniform(30, 60)
        dx = cv2.GaussianBlur(
            np.random.randn(h, w).astype(np.float32), (0, 0), sigma
        ) * alpha
        dy = cv2.GaussianBlur(
            np.random.randn(h, w).astype(np.float32), (0, 0), sigma
        ) * alpha
        x, y   = np.meshgrid(np.arange(w), np.arange(h))
        map_x  = np.clip(x + dx, 0, w - 1).astype(np.float32)
        map_y  = np.clip(y + dy, 0, h - 1).astype(np.float32)
        image  = cv2.remap(image, map_x, map_y, cv2.INTER_LINEAR)
        mask   = cv2.remap(mask,  map_x, map_y, cv2.INTER_NEAREST)

    # Coarse dropout — randomly blank out patches (simulates ultrasound artefacts)
    if random.random() > 0.5:
        h, w     = image.shape[:2]
        n_holes  = random.randint(1, 4)
        hole_size = random.randint(16, 40)
        for _ in range(n_holes):
            x1 = random.randint(0, w - hole_size)
            y1 = random.randint(0, h - hole_size)
            image[y1:y1+hole_size, x1:x1+hole_size] = 0

    return image, mask
    

class BreastUltrasoundDataset(Dataset):

    def __init__(self, image_size=256, augment=True):

        self.root_dir = kagglehub.dataset_download(
            "aryashah2k/breast-ultrasound-images-dataset"
        )

        self.image_size = image_size
        self.samples = []
        self.do_augment = augment
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
        classes = ["benign", "malignant"]

        for cls in classes:

            class_dir = os.path.join(
                self.root_dir,
                "Dataset_BUSI_with_GT",
                cls
            )

            files = os.listdir(class_dir)

            image_files = [
                f for f in files
                if "_mask" not in f and f.endswith(".png")
            ]

            for img_file in image_files:

                img_path = os.path.join(class_dir, img_file)

                base_name = img_file.replace(".png", "")

                mask_files = [
                    f for f in files
                    if f.startswith(base_name + "_mask")
                    and f.endswith(".png")
                ]

                mask_paths = [
                    os.path.join(class_dir, f)
                    for f in mask_files
                ]

                self.samples.append({
                    "image": img_path,
                    "masks": mask_paths,
                    "label": cls
                })

    def __len__(self):

        return len(self.samples)

    def pad_to_square(self, image, mask=None):

        h, w = image.shape[:2]

        size = max(h, w)

        pad_h = size - h
        pad_w = size - w

        top = pad_h // 2
        bottom = pad_h - top

        left = pad_w // 2
        right = pad_w - left

        image = cv2.copyMakeBorder(
            image,
            top,
            bottom,
            left,
            right,
            borderType=cv2.BORDER_CONSTANT,
            value=0
        )

        if mask is not None:

            mask = cv2.copyMakeBorder(
                mask,
                top,
                bottom,
                left,
                right,
                borderType=cv2.BORDER_CONSTANT,
                value=0
            )

            return image, mask

        return image

    def combine_masks(self, mask_paths, image_shape): 

        if len(mask_paths) == 0:

            return np.zeros(image_shape, dtype=np.uint8)

        combined_mask = np.zeros(image_shape, dtype=np.uint8)

        for path in mask_paths:

            mask = cv2.imread(
                path,
                cv2.IMREAD_GRAYSCALE
            )

            if mask is None:
                continue

            mask = (mask > 0).astype(np.uint8)

            combined_mask = np.maximum(
                combined_mask,
                mask
            )

        return combined_mask


    def __getitem__(self, idx):

        sample = self.samples[idx]

        image = cv2.imread(sample["image"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Failed to load image: {sample['image']}")

        mask = self.combine_masks(sample["masks"], image.shape)

        image, mask = self.pad_to_square(image, mask)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        
        #CLAHE Preprocessing
        image = self.clahe.apply(image)
        
        if self.do_augment:
            image, mask = augment(image, mask)
            
        # normalize image
        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)


        mask = mask[None, :, :]   # (1, H, W)

        image = image[None, :, :] # (1, H, W)

        return torch.from_numpy(image).float(), torch.from_numpy(mask).float()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):

    def __init__(self, smooth=1.0):

        super().__init__()

        self.smooth = smooth

    def forward(self, preds, targets):

        preds   = torch.sigmoid(preds)
        preds   = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (preds * targets).sum(dim=1)

        dice = (
            2. * intersection + self.smooth
        ) / (
            preds.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )

        return 1 - dice.mean()


class FocalLoss(nn.Module):

    def __init__(self, alpha=0.5, gamma=2):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        bce = F.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none'
        )

        pt    = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce

        return focal.mean()

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        # alpha = FP weight, beta = FN weight
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        preds = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        TP = (preds * targets).sum(dim=1)
        FP = (preds * (1 - targets)).sum(dim=1)
        FN = ((1 - preds) * targets).sum(dim=1)
        tversky = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return 1 - tversky.mean()


dice_loss  = DiceLoss()
focal_loss = FocalLoss()
tversky_loss = TverskyLoss() #Penalize FN

def criterion(preds, targets):

    dice  = dice_loss(preds, targets)
    focal = focal_loss(preds, targets)
    tversky = tversky_loss(preds, targets)
    return 0.7 * tversky +  0.3 * focal



In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
dataset = BreastUltrasoundDataset()
unet = UNet(1).to(device)


In [ ]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

train_dataset = BreastUltrasoundDataset(augment=True)
val_dataset   = BreastUltrasoundDataset(augment=False)
indices = list(range(len(train_dataset)))
labels = [s["label"] for s in train_dataset.samples]
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(Subset(val_dataset,   val_idx),   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

In [ ]:
decoder_params = [p for n, p in unet.named_parameters() if "encoder" not in n]

optimizer = torch.optim.AdamW([
    {"params": unet.encoder.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    {"params": decoder_params,            "lr": 1e-4, "weight_decay": 3e-3},  # stronger decoder decay
])

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=10
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=190, eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[10]
)

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
# checkpoint = torch.load("/kaggle/working/best_unet.pth", map_location=device)
# unet.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# scaler.load_state_dict(checkpoint["scaler_state_dict"])
# start_epoch = checkpoint["epoch"] + 1
# best_val    = checkpoint["best_val_dice_loss"]

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=200, eta_min=1e-6
# )
# scheduler.step(start_epoch) 

In [ ]:
import torch.nn.functional as F

dice_criterion = DiceLoss()

num_epochs   = 200
patience     = 60
counter      = 0
train_losses = []
val_losses   = []
best_val     = 1

for epoch in range(num_epochs):

    # ===================== TRAIN =====================
    unet.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            outputs = unet(images)
            loss    = criterion(outputs, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ===================== VALIDATION =====================
    unet.eval()
    val_loss = 0
    dice     = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device)
            with torch.amp.autocast("cuda"):
                outputs = unet(images)
            val_loss += criterion(outputs, masks).item()
            dice     += dice_criterion(outputs, masks).item()

    avg_val_loss  = val_loss / len(val_loader)
    avg_dice_loss = dice     / len(val_loader)
    val_losses.append(avg_val_loss)

    if avg_dice_loss < best_val:
        best_val = avg_dice_loss
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     unet.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict":    scaler.state_dict(),
            "best_val_dice_loss":   best_val,
        }, "/kaggle/working/best_unet.pth")
        print(f"  Saved best model at epoch {epoch+1}")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

    # ===================== LOG =====================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train: {avg_train_loss:.4f} "
        f"Val: {avg_val_loss:.4f} "
        f"Dice Score: {1 - avg_dice_loss:.4f} "
        f"LR: {current_lr:.2e}"
    )

In [ ]:
torch.save(unet.state_dict(), "model.bin")